In [ ]:
#| default_exp genetic_comparison

## What is being compared?

`compare_genetic` compares **24-bit-per-coordinate fixed-point GA**, **raw IEEE-754 GA**, **guarded IEEE-754 GA**, and a small, domain-agnostic **DE/rand/1/bin differential-evolution baseline**. The baseline is a NumPy benchmark helper, not a new optimiser-framework API, and does not require SciPy. All methods minimise the same scalar `objective(x, y)` within the same finite rectangle `start +/- max_offset`.

This is a **bounded search experiment**, not a comparison over the entire float64 bit space. Raw and guarded IEEE proposals outside the rectangle or containing NaN/infinity are rejected before objective evaluation; their bit codec is never clipped. DE uses unit-box coordinates with donor clipping to `[0, 1]`, scale factor `F=0.8`, crossover probability `CR=0.9`, and at least one donor coordinate. Trial coordinates are also clipped to physical bounds after rescaling to prevent endpoint-rounding overshoot. It caches valid parent losses and updates its population only after a complete generation. No objective-specific gradients or geometry are used.

For each seed, initial sampling is identical: `default_rng(seed).random((population_size, 2))`, scaled to the common rectangle, then the supplied start replaces row zero. Fixed-point subsequently quantises those coordinates; its discrepancy is at most half a grid step, up to floating-point rounding. The two IEEE modes and DE start at exactly the same physical coordinates.

An expected **0.48 bit flips per offspring** is the default for every GA: mutation probability is `expected_mutations / 48` for fixed-point and `/ 128` for either IEEE mode. Guarded field weights are normalised by the GA to preserve that expectation; rejection and retries can still change the realised work and displacement. DE has no bit-flip metric.

**This is an end-to-end optimiser comparison, not an isolated encoding experiment.** Fixed-point uses fitness-proportional selection in its proportional-selection slots, whereas the IEEE modes use rank-based selection there. Fixed-point also evaluates elites again with the next population; IEEE caches elite fitness from the current parent evaluation. Both GA families re-evaluate parents on each update, while DE caches unchanged parents. These differences in selection pressure, validation, operators and evaluation policy are part of the methods being compared. Matching bit-flip expectations and objective-call budgets does not remove those differences.

The budget counts **every actual objective invocation**, including initial populations, GA parent rechecks, accepted/rejected evaluated offspring, and objective-valued retries. Invalid chromosomes rejected before the objective do not consume that budget. Nothing is memoised by the comparison wrapper. Each curve has exactly `evaluation_budget` entries, indexed by objective calls rather than generations; an internal stop signal prevents the next call. Equal call budgets do not imply equal wall-clock costs.

Objectives should be deterministic and side-effect-free for a scientific comparison. User exceptions and non-scalar/non-real results propagate; nonfinite initial losses fail explicitly. IEEE and DE nonfinite offspring losses consume a call and are rejected without improving best-so-far. The fixed-point baseline retains its existing strict finite-loss contract: a nonfinite result at any evaluation raises `ValueError`, so use an objective finite throughout the search rectangle for a complete four-method comparison. No method is assumed universally best, and guarded IEEE is not claimed to dominate raw IEEE or ordinary real-valued search.

In [ ]:
#| exporti
from dataclasses import dataclass
from numbers import Integral, Real

import numpy as np

from optimisations.genetic import Diagnostics, genetic_algo

In [ ]:
#| export
@dataclass(frozen=True)
class ComparisonResult:
    """One seeded run; its read-only best-loss curve includes partial generations.

    Population diversity (fraction of distinct coordinate pairs), generation
    count and diagnostics describe only the last completed state/updates.
    The final interrupted update is not included in diagnostics_history.
    DE diagnostics exclude its initial population evaluation, have no exponent
    histogram, and count invalid objective proposals, not non-improving trials.
    Their shared mutation_displacement field is an unused zero; the result's
    mutation_displacement property returns None for DE.
    """

    seed: int
    method: str
    evaluations: int
    best_loss: float
    best_params: tuple[float, float]
    best_loss_history: np.ndarray
    nonfinite_evaluations: int
    completed_generations: int
    population_diversity: float
    diagnostics_history: tuple[Diagnostics, ...]

    @property
    def completed_evaluations(self):
        """Objective calls recorded by completed-update diagnostics only."""
        return sum(item.evaluations for item in self.diagnostics_history)

    @property
    def rejection_rate(self):
        """Invalid proposals / attempted offspring across completed updates."""
        attempted = sum(item.offspring for item in self.diagnostics_history)
        rejected = sum(item.rejected for item in self.diagnostics_history)
        return rejected / attempted if attempted else 0.0

    @property
    def mutation_displacement(self):
        """Mean completed-update GA displacement; None for DE or no updates."""
        if self.method == 'differential_evolution' or not self.diagnostics_history:
            return None
        return float(np.mean([item.mutation_displacement for item in self.diagnostics_history]))

    @property
    def exponent_histogram(self):
        """Last completed update's raw biased-exponent counts, if available."""
        return self.diagnostics_history[-1].exponent_histogram if self.diagnostics_history else ()

In [ ]:
#| exporti
class _EvaluationLimit(Exception):
    pass


class _BudgetObjective:
    def __init__(self, objective, limit):
        self.objective = objective
        self.limit = limit
        self.evaluations = 0
        self.nonfinite_evaluations = 0
        self.best_loss = np.inf
        self.best_params = None
        self.history = []

    def __call__(self, x, y):
        if self.evaluations >= self.limit:
            raise _EvaluationLimit
        self.evaluations += 1
        loss = np.asarray(self.objective(x, y))
        if loss.shape != () or loss.dtype.kind not in 'iuf':
            raise ValueError('objective(x, y) must return a real scalar')
        value = float(loss)
        if not np.isfinite(value):
            self.nonfinite_evaluations += 1
        elif value < self.best_loss:
            self.best_loss = value
            self.best_params = (float(x), float(y))
        self.history.append(self.best_loss)
        return value


def _comparison_result(seed, method, tracked, population, diagnostics):
    curve = np.asarray(tracked.history, dtype=np.float64)
    curve.setflags(write=False)
    return ComparisonResult(
        seed=seed, method=method, evaluations=tracked.evaluations,
        best_loss=tracked.best_loss, best_params=tracked.best_params,
        best_loss_history=curve, nonfinite_evaluations=tracked.nonfinite_evaluations,
        completed_generations=len(diagnostics),
        population_diversity=len(np.unique(population, axis=0)) / len(population),
        diagnostics_history=tuple(diagnostics))


def _integer_option(value, name, minimum):
    if (isinstance(value, (bool, np.bool_)) or not isinstance(value, Integral)
            or value < minimum):
        raise ValueError(f'{name} must be an integer >= {minimum}')
    return int(value)


def _comparison_options(objective, start, seeds, evaluation_budget, population_size,
                        max_offset, expected_mutations, max_retries):
    if not callable(objective):
        raise ValueError('objective must be callable as objective(x, y)')
    population_size = _integer_option(population_size, 'population_size', 4)
    evaluation_budget = _integer_option(evaluation_budget, 'evaluation_budget', population_size)
    if (isinstance(expected_mutations, (bool, np.bool_))
            or not isinstance(expected_mutations, Real)
            or not 0 <= expected_mutations <= 48):
        raise ValueError('expected_mutations must be a finite real number in [0, 48]')
    expected_mutations = float(expected_mutations)
    try:
        seeds = tuple(seeds)
    except TypeError as error:
        raise ValueError('seeds must be a nonempty iterable of nonnegative integers') from error
    if not seeds:
        raise ValueError('seeds must be a nonempty iterable of nonnegative integers')
    seeds = tuple(_integer_option(seed, 'seed', 0) for seed in seeds)
    # Reuse the public GA's validation and bounds policy before any objective call.
    init, _, _ = genetic_algo(population_size, max_offset,
                              mutation_chance=expected_mutations / 48,
                              seed=0, max_retries=max_retries)
    bounds = init(start).bounds.copy()
    start = np.asarray(start, dtype=np.float64).copy()
    return start, bounds, seeds, evaluation_budget, population_size, expected_mutations

In [ ]:
#| exporti
def _run_ga(tracked, start, seed, method, population_size, max_offset,
            expected_mutations, max_retries):
    encoding = 'fixed' if method == 'fixed' else 'ieee754'
    operators = 'guarded' if method == 'ieee754_guarded' else 'standard'
    bits = 48 if encoding == 'fixed' else 128
    init, update, get_params = genetic_algo(
        population_size, max_offset, mutation_chance=expected_mutations / bits,
        seed=seed, encoding=encoding, operators=operators, max_retries=max_retries)
    state = init(start)
    diagnostics = []
    try:
        while tracked.evaluations < tracked.limit:
            state = update(len(diagnostics), tracked, state)
            diagnostics.append(state.diagnostics)
    except _EvaluationLimit:
        pass
    return _comparison_result(seed, method, tracked, get_params(state), diagnostics)


def _run_differential_evolution(tracked, start, bounds, seed, population_size):
    rng = np.random.default_rng(seed)
    width = bounds[1] - bounds[0]
    unit_population = rng.random((population_size, 2))
    population = bounds[0] + unit_population * width
    population[0] = start
    unit_population[0] = np.divide(start - bounds[0], width,
                                   out=np.zeros(2), where=width != 0)
    losses = np.empty(population_size)
    for index, (x, y) in enumerate(population):
        losses[index] = tracked(x, y)
        if not np.isfinite(losses[index]):
            raise ValueError('initial objective(x, y) must return a finite real scalar')
    diagnostics = []
    try:
        while tracked.evaluations < tracked.limit:
            next_unit, next_population, next_losses = (
                unit_population.copy(), population.copy(), losses.copy())
            rejected_objective = 0
            for index in range(population_size):
                available = np.delete(np.arange(population_size), index)
                r1, r2, r3 = rng.choice(available, 3, replace=False)
                donor = unit_population[r1] + 0.8 * (unit_population[r2] - unit_population[r3])
                donor = np.clip(donor, 0, 1)
                cross = rng.random(2) < 0.9
                cross[rng.integers(2)] = True
                trial_unit = np.where(cross, donor, unit_population[index])
                trial = np.clip(bounds[0] + trial_unit * width, bounds[0], bounds[1])
                loss = tracked(*trial)
                if not np.isfinite(loss):
                    rejected_objective += 1
                elif loss <= losses[index]:
                    next_unit[index], next_population[index], next_losses[index] = trial_unit, trial, loss
            unit_population, population, losses = next_unit, next_population, next_losses
            diagnostics.append(Diagnostics(
                evaluations=population_size, offspring=population_size,
                rejected_objective=rejected_objective, retained_parents=rejected_objective))
    except _EvaluationLimit:
        pass
    return _comparison_result(seed, 'differential_evolution', tracked, population, diagnostics)

In [ ]:
#| export
def compare_genetic(objective, start, *, seeds=(0, 1, 2), evaluation_budget=2000,
                    population_size=50, max_offset=4, expected_mutations=0.48,
                    max_retries=8):
    """Compare three GA representations and NumPy DE under exact call budgets.

    Return one ComparisonResult per seed/method, in seed order then fixed,
    ieee754, ieee754_guarded, differential_evolution order. Seeds must be a
    nonempty finite iterable of nonnegative integers. Every run shares the
    same bounded initial sampling (apart from fixed-point quantisation).

    population_size must be >= 4; evaluation_budget >= population_size.
    expected_mutations must lie in [0, 48]. Objective calls and best curves
    include interrupted updates; population/diagnostics exclude them.
    Nonfinite initial losses and invalid configuration raise ValueError;
    fixed-point additionally requires finite losses at every evaluation.
    User objective exceptions propagate unchanged.
    """
    start, bounds, seeds, evaluation_budget, population_size, expected_mutations = _comparison_options(
        objective, start, seeds, evaluation_budget, population_size,
        max_offset, expected_mutations, max_retries)
    results = []
    for seed in seeds:
        for method in ('fixed', 'ieee754', 'ieee754_guarded', 'differential_evolution'):
            tracked = _BudgetObjective(objective, evaluation_budget)
            if method == 'differential_evolution':
                result = _run_differential_evolution(tracked, start, bounds, seed, population_size)
            else:
                result = _run_ga(tracked, start, seed, method, population_size, max_offset,
                                 expected_mutations, max_retries)
            results.append(result)
    return results

## Quick reproducible run

This small example intentionally uses few calls. It is a smoke test of the comparison protocol, not statistical evidence about which representation wins. **A zero-centred sphere is especially favourable to the IEEE representation:** exponent changes can produce extremely small coordinates directly. Squaring sufficiently small, nonzero float64 coordinates underflows, so an observed objective value of exactly zero is **not proof of the exact mathematical minimiser**. Inspect `best_params` as well as loss; do not interpret underflow-induced zero as convergence to exactly `(0, 0)` or evidence of universal superiority. The translated/scaled suite below is essential for exposing this origin and scale sensitivity.

Curves and `best_params` include the last partially evaluated generation. In contrast, `diagnostics_history`, `completed_generations`, `population_diversity`, `rejection_rate`, `mutation_displacement` and `exponent_histogram` include **completed updates only**; they must not be treated as counters for the whole truncated budget.

Diversity is the fraction of unique decoded coordinate pairs in the last completed population (or initial population if no update completes). GA displacement is the mean normalised operator displacement from its selected parent, averaged across completed updates. Exponent histograms use raw biased IEEE fields (`0` includes zero/subnormals). DE has no exponent histogram or bit-mutation displacement. Its diagnostic evaluations exclude initial sampling, whereas every completed GA update includes its parent re-evaluations.

In [ ]:
def sphere(x, y):
    return x * x + y * y

quick = compare_genetic(sphere, [3.0, -3.0], seeds=(0, 1),
                        evaluation_budget=100, population_size=10)
[(run.seed, run.method, run.best_loss, run.evaluations) for run in quick]

In [ ]:
#| include: false
assert len(quick) == 8
assert all(run.evaluations == len(run.best_loss_history) == 100 for run in quick)
assert all(np.all(np.diff(run.best_loss_history) <= 0) for run in quick)
assert all(sphere(*run.best_params) == run.best_loss for run in quick)

## Translated and scaled objectives: optional extended experiment

The following cells are disabled during routine documentation execution (`eval: false`). Run them manually for ten seeds, 2,000 calls per method and transformed sphere/Himmelblau objectives. Both coordinate scales and translations change while the normalised domain, start, optimum loss and sampling protocol remain the same; this probes representation sensitivity rather than giving one method a different search region. All transformations are generic wrappers around ordinary NumPy callables.

Report means **and** medians across seeds, inspect convergence bands, and compare completed-update rejection/diversity diagnostics. Results are illustrative, not a universal ranking: choices of domain, encoding, clipping, selection, mutation expectation and seed set matter. In particular, guarded IEEE need not outperform raw IEEE, fixed-point or DE. Objective-call fairness intentionally charges fixed-point elite rechecks and all GA parent rechecks, while IEEE caches elites and DE caches unchanged parents. Rejections may also impose substantial uncounted runtime overhead. The sphere's exact-zero floating-point losses can reflect underflow, not exact zero coordinates; translations and scales are therefore necessary checks, not optional evidence to omit when reporting a favourable zero-centred result.

In [ ]:
#| eval: false
def himmelblau(x, y):
    return (x * x + y - 11) ** 2 + (x + y * y - 7) ** 2

def translated_scaled(base, shift, scale):
    shift = np.asarray(shift, dtype=float)
    def objective(x, y):
        return base((x - shift[0]) / scale, (y - shift[1]) / scale)
    return objective

experiments = {}
transforms = [('ordinary', (0.0, 0.0), 1.0),
              ('translated', (100.0, -80.0), 1.0),
              ('small', (0.0, 0.0), 1e-3),
              ('large_translated', (1e5, -2e5), 1e3)]
for name, base in [('sphere', sphere), ('himmelblau', himmelblau)]:
    for variant, shift, scale in transforms:
        objective = translated_scaled(base, shift, scale)
        start = np.asarray(shift) + scale * np.array([3.0, -3.0])
        experiments[f'{name}/{variant}'] = compare_genetic(
            objective, start, max_offset=4 * scale, seeds=range(10),
            evaluation_budget=2000, population_size=50)

In [ ]:
#| eval: false
for name, runs in experiments.items():
    print(name)
    for method in dict.fromkeys(run.method for run in runs):
        selected = [run for run in runs if run.method == method]
        losses = [run.best_loss for run in selected]
        rejected = np.mean([run.rejection_rate for run in selected])
        diversity = np.mean([run.population_diversity for run in selected])
        movement = [run.mutation_displacement for run in selected
                    if run.mutation_displacement is not None]
        print(f'  {method:24} mean={np.mean(losses):.6g} median={np.median(losses):.6g}'
              f' completed rejection={rejected:.3f} diversity={diversity:.3f}'
              f' GA displacement={np.mean(movement) if movement else None}')
        if selected[0].exponent_histogram:
            print('    seed-0 final completed exponent histogram:', selected[0].exponent_histogram)

In [ ]:
#| eval: false
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(18, 8), squeeze=False)
for ax, (name, runs) in zip(axes.flat, experiments.items()):
    for method in dict.fromkeys(run.method for run in runs):
        curves = np.stack([run.best_loss_history for run in runs if run.method == method])
        calls = np.arange(1, curves.shape[1] + 1)
        q25, median, q75 = np.quantile(curves, [0.25, 0.5, 0.75], axis=0)
        line, = ax.plot(calls, median, label=method)
        ax.fill_between(calls, q25, q75, color=line.get_color(), alpha=0.15)
    ax.set(title=name, xlabel='actual objective calls', ylabel='best loss')
    ax.set_yscale('symlog', linthresh=1e-8)
axes.flat[0].legend()
fig.tight_layout()